# Prepare Final Weather Dataset

Notebook chuẩn bị dataset cuối cùng cho EDA và modeling.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/processed/Hanoi_clean.csv").exists() and (PROJECT_ROOT.parent / "data/processed/Hanoi_clean.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_PATH = PROJECT_ROOT / "data/processed/Hanoi_clean.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/Hanoi_final.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(CLEAN_PATH)
print(OUTPUT_PATH)

## 1. Đọc dữ liệu sạch

In [ ]:
df = pd.read_csv(CLEAN_PATH)
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
df.head()

## 2. Thêm biến lịch cơ bản

In [ ]:
df["hour"] = df["time"].dt.hour
df["day_of_week"] = df["time"].dt.dayofweek
df["day_of_month"] = df["time"].dt.day
df["day_of_year"] = df["time"].dt.dayofyear
df["month"] = df["time"].dt.month
df["year"] = df["time"].dt.year
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["day_of_year_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 366)
df["day_of_year_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 366)

df.head()

## 3. Bỏ cột hằng không hữu ích

In [ ]:
constant_columns = []
for column in df.columns:
    if column in ["city", "latitude", "longitude"]:
        continue
    if df[column].nunique(dropna=False) <= 1:
        constant_columns.append(column)

print("Constant columns:", constant_columns)
df = df.drop(columns=constant_columns)

## 4. Kiểm tra dataset cuối

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(dropna=False),
})

summary.head(40)

## 5. Lưu final dataset

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")